# 04 · Dynamic Programming
**TABULAR METHODS** · ⏱ 50 min · 🧪 3 exercises · ⭐ 400 XP

*Planning with a perfect model — policy evaluation, policy improvement, policy iteration, value iteration and GPI.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "04-dynamic-programming"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

The previous lesson ended with two equations, and with a problem: the Bellman expectation
equation is linear and can be solved directly, but only for a fixed policy, and only at O(n³)
cost. The optimality equation has a max in it and cannot be solved in one step at all.

**Dynamic programming (DP)** is the family of algorithms that solves both equations by
**turning them into update rules** and applying them over and over until nothing changes. DP
needs a perfect model of the environment, the `env.P` you have been using, so it is
*planning*, not learning. It matters for two reasons:

1. On small problems it gives you the **exact** answer, which you will use as ground truth to
   test every learning algorithm in later lessons.
2. Every RL algorithm afterwards (Monte Carlo, TD, Q-learning, DQN, actor-critic) is an
   attempt to do what DP does **without the model** and **without visiting every state**.

## 01 · Iterative policy evaluation

Take the Bellman expectation equation and read the "=" as "←":

```text
Vₖ₊₁(s) ← Σₐ π(a|s) Σₛ',ᵣ p(s', r | s, a) [ r + γ Vₖ(s') ]
```

Start from any V₀ (zeros are fine). Each full pass over the states is a **sweep**. As k → ∞,
Vₖ → v_π. Why it works: with γ < 1 each sweep shrinks the worst-case error by at least a factor of
γ, because the update is a **contraction**. Errors in the future values are discounted before
they feed back.

We cannot sweep forever, so we stop when the largest change in a sweep, Δ, falls below a small
threshold θ.

> **ℹ️ NOTE** · **Two copies or one?**
>
> The textbook equation uses two arrays: compute all Vₖ₊₁ from the old Vₖ (a *synchronous*
> sweep). Most implementations use **one array updated in place**: states later in the sweep
> immediately see the new values of earlier states. In-place sweeps converge to the same
> answer, usually faster. All exercises below use in-place sweeps.

Worked example: corridor `S . G`, uniform random policy, +1 for entering G, γ = 0.9, V₀ = 0.
Sweep 1 updates S first: every outcome leads to a state worth 0 and pays 0, so V(S) stays 0. Then
the middle cell: ¼ of the time it enters G for +1, so V(M) = 0.25. Sweep 2: S now sees V(M), so
V(S) = 0.9·(¾·0 + ¼·0.25) ≈ 0.056. Value **spreads backward from the reward one step per sweep**,
and after enough sweeps it settles at the exact answer from the last lesson,
v = [0.439, 0.634, 0].

Watch the same thing on a 4 × 3 world. Each click of *step* is one sweep of policy evaluation
for the uniform random policy:

In [ ]:
#@title 🎮 **Interactive** · Dynamic programming sandbox
rlcolab.widget("gridworld", map=["...G", ".#.X", "S..."], gamma=0.9, step=-0.04, slip=0.2, mode="policy-evaluation")

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("04-dynamic-programming#0", {"q": "In iterative policy evaluation, why do the values near the goal settle before the values far from it?", "options": ["Those states are updated more often", "Reward information propagates backward one transition per sweep", "θ is smaller near the goal", "The policy is better near the goal"], "answer": 1, "explain": "Each backup only looks one step ahead, so a state k steps from the reward first 'hears' about it in sweep k. Far states need more sweeps."})

---
### 🧪 Exercise 1 · Iterative policy evaluation · ⭐ **120 XP**

Compute v_π by turning the Bellman expectation equation into an update rule.

1. `expected_update(env, V, s, policy, gamma)` — return `Σₐ π(a|s) Σ prob · (r + γ·V[s'])` over `env.P[s][a]`, dropping `γ·V[s']` when the transition terminates.
2. `policy_evaluation(env, policy, gamma, theta, max_sweeps, V=None)` — start from zeros (or the given `V`), and sweep over **all** states, writing each new value into `V` immediately (in place). Track `delta`, the biggest change in the sweep, and stop once `delta < theta` or after `max_sweeps`. Return `(V, sweeps)`.

The check compares your answer with the exact linear solve from the previous lesson. **Run** shows values spreading out from the terminals sweep by sweep, and what each extra digit of accuracy costs.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 04-dynamic-programming/01-policy-evaluation
import numpy as np
import rlhud
from rlenvs import GridWorld


def expected_update(env, V, s, policy, gamma):
    """The Bellman expectation backup for one state:

        Σ_a π(a|s) Σ_{s',r} p(s',r|s,a) [ r + γ V(s') ]     (no γ·V(s') after termination)

    policy[s, a] = π(a|s).
    """
    # TODO: weight every outcome of every action by π(a|s)·prob
    raise NotImplementedError


def policy_evaluation(env, policy, gamma, theta=1e-8, max_sweeps=10_000, V=None):
    """Iterative policy evaluation, in place.

    Sweep over all states, replacing V[s] by expected_update(...) immediately, and track
    Δ = the largest |change| in the sweep. Stop as soon as Δ < theta.
    Returns (V, sweeps) where sweeps is the number of full sweeps performed.
    """
    V = np.zeros(env.n_states) if V is None else np.array(V, dtype=float)
    sweep = 0
    # TODO: repeat sweeps (at most max_sweeps); in each, update every state in place
    #       and track delta; stop when delta < theta
    raise NotImplementedError
    return V, sweep


if __name__ == "__main__":
    env = GridWorld(["....G",
                     ".##.X",
                     "....#",
                     "S...."], slip=0.1, step_reward=-0.04)
    gamma = 0.9
    random_pi = np.full((env.n_states, 4), 0.25)

    # Watch values spread outward from the terminals, one sweep at a time
    V = np.zeros(env.n_states)
    for k in [1, 2, 3]:
        V, _ = policy_evaluation(env, random_pi, gamma, max_sweeps=1, V=V)
        rlhud.grid(env.to_grid(V), title=f"Random policy · after {k} sweep(s)",
                   walls=env.walls_mask(), labels=env.labels())

    V, sweeps = policy_evaluation(env, random_pi, gamma, theta=1e-8)
    rlhud.grid(env.to_grid(V), title=f"v_random converged ({sweeps} sweeps)",
               walls=env.walls_mask(), labels=env.labels())

    # How many sweeps does each accuracy cost?
    thetas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-6, 1e-8, 1e-10]
    rows = [[f"{t:g}", policy_evaluation(env, random_pi, gamma, theta=t)[1]] for t in thetas]
    rlhud.table(rows, headers=["θ", "sweeps"], title="Sweeps needed vs tolerance")
    rlhud.stat("v_random(start)", float(V[env.start]))

In [ ]:
#@title ✅ **Check** · Iterative policy evaluation (+120 XP) — tests the code you last ran above
rlcolab.check("04-dynamic-programming/01-policy-evaluation")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`expected_update` is the same double loop as `q_from_v`, except each outcome is weighted by `policy[s, a] * prob` and everything is summed into one number.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Structure: `for sweep in range(1, max_sweeps + 1): delta = 0.0; for s in range(env.n_states): v_new = expected_update(...); delta = max(delta, abs(v_new - V[s])); V[s] = v_new` and then `if delta < theta: break`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Return `V, sweep`. Make sure `V` is a float array you are allowed to modify: `V = np.zeros(env.n_states) if V is None else np.array(V, dtype=float)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("04-dynamic-programming/01-policy-evaluation")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 04-dynamic-programming/01-policy-evaluation
# Reference solution
import numpy as np
import rlhud
from rlenvs import GridWorld


def expected_update(env, V, s, policy, gamma):
    """The Bellman expectation backup for one state:

        Σ_a π(a|s) Σ_{s',r} p(s',r|s,a) [ r + γ V(s') ]     (no γ·V(s') after termination)

    policy[s, a] = π(a|s).
    """
    total = 0.0
    for a in range(env.n_actions):
        if policy[s, a] == 0:
            continue
        for prob, s2, r, done in env.P[s][a]:
            total += policy[s, a] * prob * (r + (0.0 if done else gamma * V[s2]))
    return total


def policy_evaluation(env, policy, gamma, theta=1e-8, max_sweeps=10_000, V=None):
    """Iterative policy evaluation, in place.

    Sweep over all states, replacing V[s] by expected_update(...) immediately, and track
    Δ = the largest |change| in the sweep. Stop as soon as Δ < theta.
    Returns (V, sweeps) where sweeps is the number of full sweeps performed.
    """
    V = np.zeros(env.n_states) if V is None else np.array(V, dtype=float)
    for sweep in range(1, max_sweeps + 1):
        delta = 0.0
        for s in range(env.n_states):
            v_new = expected_update(env, V, s, policy, gamma)
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta:
            break
    return V, sweep


if __name__ == "__main__":
    env = GridWorld(["....G",
                     ".##.X",
                     "....#",
                     "S...."], slip=0.1, step_reward=-0.04)
    gamma = 0.9
    random_pi = np.full((env.n_states, 4), 0.25)

    # Watch values spread outward from the terminals, one sweep at a time
    V = np.zeros(env.n_states)
    for k in [1, 2, 3]:
        V, _ = policy_evaluation(env, random_pi, gamma, max_sweeps=1, V=V)
        rlhud.grid(env.to_grid(V), title=f"Random policy · after {k} sweep(s)",
                   walls=env.walls_mask(), labels=env.labels())

    V, sweeps = policy_evaluation(env, random_pi, gamma, theta=1e-8)
    rlhud.grid(env.to_grid(V), title=f"v_random converged ({sweeps} sweeps)",
               walls=env.walls_mask(), labels=env.labels())

    # How many sweeps does each accuracy cost?
    thetas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-6, 1e-8, 1e-10]
    rows = [[f"{t:g}", policy_evaluation(env, random_pi, gamma, theta=t)[1]] for t in thetas]
    rlhud.table(rows, headers=["θ", "sweeps"], title="Sweeps needed vs tolerance")
    rlhud.stat("v_random(start)", float(V[env.start]))

## 02 · Policy improvement

Knowing v_π is only useful if it helps us find a better policy. Suppose that in state *s* we
deviate from π just once: take action *a*, then follow π. That is worth exactly q_π(s, a). If
some action has q_π(s, a) > v_π(s), deviating to it in *s* is better than following π there.

> **🔑 KEY IDEA** · **The policy improvement theorem**
>
> If π' is any policy with q_π(s, π'(s)) ≥ v_π(s) in every state, then v_π'(s) ≥ v_π(s) in
> every state. In particular, the **greedy** policy
> π'(s) = argmaxₐ q_π(s, a) = argmaxₐ Σₛ',ᵣ p(s', r | s, a)[ r + γ v_π(s') ]
> is at least as good as π everywhere, and strictly better somewhere unless π is already optimal.

The proof is a short unrolling: since choosing π'(s) once is at least as good as following
π, apply the same argument at the next state, and the next. Each step can only gain. Choosing
better actions *everywhere* therefore never hurts.

And when does improvement stop? When greedy improvement returns the same policy, then
v_π(s) = maxₐ Σ p [r + γ v_π(s')] for all s. That is the **Bellman optimality equation**, so
π is optimal. Improvement either makes strict progress or proves you are done.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("04-dynamic-programming#1", {"q": "Greedy improvement of the uniform random policy gives π'. What can you say for sure?", "options": ["π' is optimal", "v_π'(s) ≥ v_random(s) in every state", "v_π' is better only at the start state", "Nothing until π' is evaluated"], "answer": 1, "explain": "The policy improvement theorem guarantees π' is at least as good everywhere. Optimal usually needs more rounds of evaluation and improvement."})

## 03 · Policy iteration

Alternate the two steps:

```text
π₀ →(evaluate) v_π₀ →(improve) π₁ →(evaluate) v_π₁ →(improve) π₂ → … → π*
```

Each policy is strictly better than the last (or optimal), and a finite MDP has finitely many
deterministic policies, so **policy iteration terminates in a finite number of iterations**.
In practice the number is tiny: often fewer than ten even for large gridworlds, because each
improvement step changes the action in every state at once. Compare that with the
4096 policies you enumerated by brute force in the last lesson.

Two practical details make policy iteration fast and reliable:

- **Warm start**: evaluate each new policy starting from the previous policy's V, not from zeros.
  Consecutive policies have similar values, so evaluation needs only a few sweeps.
- **Ties**: if two actions are equally good, an implementation that re-picks between them
  arbitrarily can flip forever and never report "stable". Keep the old action unless a strictly
  better one exists.

> **💡 TIP** · **The 4×3 world is a classic**
>
> `GridWorld(["...G", ".#.X", "S..."], slip=0.2, step_reward=-0.04)` is the world from Russell &
> Norvig's *AI: A Modern Approach*. With γ = 1 your values should match their famous table,
> each shifted by +0.04, because their convention also charges −0.04 for the state you start in
> (for example v*(start) = 0.745 here vs 0.705 in the book).
> Look at the bottom row: the optimal policy takes the long way round rather than walk up past
> the pit, where a 10% slip would cost −1.

---
### 🧪 Exercise 2 · Policy iteration · ⭐ **140 XP**

Evaluate, improve, repeat. `q_from_v` and `policy_evaluation` are provided.

1. `policy_improvement(env, V, gamma, old_policy=None, tol=1e-9)` — return the **greedy** policy w.r.t. `V` as a one-hot `(n_states, n_actions)` array: in each state, the action with the largest `q_from_v(...)`. If `old_policy` is given and its action is within `tol` of the best, keep the old action.
2. `policy_iteration(env, gamma, theta)` — start from the uniform random policy and alternate `V = policy_evaluation(env, policy, gamma, theta, V)` and `policy_improvement(..., old_policy=policy)` until the improved policy is **identical** to the old one. Return `(policy, V, iterations)`.

The check uses the classic 4×3 world from *Russell & Norvig* (slip 0.2, step −0.04) and shows how γ changes the optimal route. **Run** draws every evaluate → improve round.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 04-dynamic-programming/02-policy-iteration
import numpy as np
import rlhud
from rlenvs import GridWorld


# ---- helpers from earlier exercises (complete) ---------------------------------
def q_from_v(env, V, s, gamma):
    """One-step lookahead: array of q(s, a) for every action."""
    q = np.zeros(env.n_actions)
    for a in range(env.n_actions):
        for prob, s2, r, done in env.P[s][a]:
            q[a] += prob * (r + (0.0 if done else gamma * V[s2]))
    return q


def policy_evaluation(env, policy, gamma, theta=1e-8, V=None):
    """In-place iterative policy evaluation. policy[s, a] = π(a|s). Returns V."""
    V = np.zeros(env.n_states) if V is None else np.array(V, dtype=float)
    while True:
        delta = 0.0
        for s in range(env.n_states):
            v_new = sum(policy[s, a] * prob * (r + (0.0 if done else gamma * V[s2]))
                        for a in range(env.n_actions) for prob, s2, r, done in env.P[s][a])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta:
            return V


# ---- your code ------------------------------------------------------------------
def policy_improvement(env, V, gamma, old_policy=None, tol=1e-9):
    """Greedy policy with respect to V, as a one-hot (n_states, n_actions) array.

    In each state pick argmax_a q(s, a). If old_policy is given and its action in s is
    within `tol` of the best q-value, KEEP the old action — otherwise policy iteration can
    flip forever between equally good actions and never report "stable".
    """
    new = np.zeros((env.n_states, env.n_actions))
    # TODO: for every state, compute q with q_from_v, choose the greedy action
    #       (keeping the old one if it is within tol of the best), set new[s, action] = 1
    raise NotImplementedError
    return new


def policy_iteration(env, gamma, theta=1e-8):
    """Alternate evaluation and improvement until the policy stops changing.

    Start from the uniform random policy. Each iteration:
        V      = policy_evaluation(env, policy, gamma, theta, V)   (warm-start from the last V)
        policy = policy_improvement(env, V, gamma, old_policy=policy)
    Stop when the improved policy equals the old one.
    Returns (policy, V, iterations).
    """
    policy = np.full((env.n_states, env.n_actions), 1.0 / env.n_actions)
    V = np.zeros(env.n_states)
    iterations = 0
    # TODO: loop evaluate → improve until the policy is stable
    raise NotImplementedError


if __name__ == "__main__":
    env = GridWorld(["....G",
                     ".##.X",
                     "....#",
                     "S...."], slip=0.1, step_reward=-0.04)
    gamma = 0.95
    # Show every evaluate → improve round
    policy = np.full((env.n_states, 4), 0.25)
    V = np.zeros(env.n_states)
    values_at_start = []
    for k in range(1, 20):
        V = policy_evaluation(env, policy, gamma, 1e-8, V)
        values_at_start.append(float(V[env.start]))
        new = policy_improvement(env, V, gamma, old_policy=policy)
        rlhud.grid(env.to_grid(V), title=f"Iteration {k}: v_π and its greedy improvement",
                   id=f"pi-{k}", policy=env.policy_grid(new), walls=env.walls_mask(), labels=env.labels())
        if np.array_equal(new, policy):
            break
        policy = new
    rlhud.plot(values_at_start, label="v_π(start)", title="Policy iteration: value of start per iteration",
               xlabel="iteration")

    pi_star, V_star, iters = policy_iteration(env, gamma)
    rlhud.text(f"Policy iteration converged in {iters} iterations. v*(start) = {V_star[env.start]:.4f}",
               title="Done", tone="good")

In [ ]:
#@title ✅ **Check** · Policy iteration (+140 XP) — tests the code you last ran above
rlcolab.check("04-dynamic-programming/02-policy-iteration")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`policy_improvement`: loop over states; `q = q_from_v(env, V, s, gamma)`; `best = int(np.argmax(q))`; then `new[s, best] = 1.0`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Tie rule: `old_a = int(np.argmax(old_policy[s]))`; `if q[old_a] >= q[best] - tol: best = old_a`. Without it, two equally good actions can alternate forever and the loop never sees a stable policy.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`policy_iteration` loop: `iterations += 1; V = policy_evaluation(env, policy, gamma, theta, V); new = policy_improvement(env, V, gamma, old_policy=policy); if np.array_equal(new, policy): return policy, V, iterations; policy = new`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("04-dynamic-programming/02-policy-iteration")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 04-dynamic-programming/02-policy-iteration
# Reference solution
import numpy as np
import rlhud
from rlenvs import GridWorld


# ---- helpers from earlier exercises (complete) ---------------------------------
def q_from_v(env, V, s, gamma):
    """One-step lookahead: array of q(s, a) for every action."""
    q = np.zeros(env.n_actions)
    for a in range(env.n_actions):
        for prob, s2, r, done in env.P[s][a]:
            q[a] += prob * (r + (0.0 if done else gamma * V[s2]))
    return q


def policy_evaluation(env, policy, gamma, theta=1e-8, V=None):
    """In-place iterative policy evaluation. policy[s, a] = π(a|s). Returns V."""
    V = np.zeros(env.n_states) if V is None else np.array(V, dtype=float)
    while True:
        delta = 0.0
        for s in range(env.n_states):
            v_new = sum(policy[s, a] * prob * (r + (0.0 if done else gamma * V[s2]))
                        for a in range(env.n_actions) for prob, s2, r, done in env.P[s][a])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta:
            return V


# ---- your code ------------------------------------------------------------------
def policy_improvement(env, V, gamma, old_policy=None, tol=1e-9):
    """Greedy policy with respect to V, as a one-hot (n_states, n_actions) array.

    In each state pick argmax_a q(s, a). If old_policy is given and its action in s is
    within `tol` of the best q-value, KEEP the old action — otherwise policy iteration can
    flip forever between equally good actions and never report "stable".
    """
    new = np.zeros((env.n_states, env.n_actions))
    for s in range(env.n_states):
        q = q_from_v(env, V, s, gamma)
        best = int(np.argmax(q))
        if old_policy is not None:
            old_a = int(np.argmax(old_policy[s]))
            if q[old_a] >= q[best] - tol:
                best = old_a
        new[s, best] = 1.0
    return new


def policy_iteration(env, gamma, theta=1e-8):
    """Alternate evaluation and improvement until the policy stops changing.

    Start from the uniform random policy. Each iteration:
        V      = policy_evaluation(env, policy, gamma, theta, V)   (warm-start from the last V)
        policy = policy_improvement(env, V, gamma, old_policy=policy)
    Stop when the improved policy equals the old one.
    Returns (policy, V, iterations).
    """
    policy = np.full((env.n_states, env.n_actions), 1.0 / env.n_actions)
    V = np.zeros(env.n_states)
    iterations = 0
    while True:
        iterations += 1
        V = policy_evaluation(env, policy, gamma, theta, V)
        new_policy = policy_improvement(env, V, gamma, old_policy=policy)
        if np.array_equal(new_policy, policy):
            return policy, V, iterations
        policy = new_policy


if __name__ == "__main__":
    env = GridWorld(["....G",
                     ".##.X",
                     "....#",
                     "S...."], slip=0.1, step_reward=-0.04)
    gamma = 0.95
    # Show every evaluate → improve round
    policy = np.full((env.n_states, 4), 0.25)
    V = np.zeros(env.n_states)
    values_at_start = []
    for k in range(1, 20):
        V = policy_evaluation(env, policy, gamma, 1e-8, V)
        values_at_start.append(float(V[env.start]))
        new = policy_improvement(env, V, gamma, old_policy=policy)
        rlhud.grid(env.to_grid(V), title=f"Iteration {k}: v_π and its greedy improvement",
                   id=f"pi-{k}", policy=env.policy_grid(new), walls=env.walls_mask(), labels=env.labels())
        if np.array_equal(new, policy):
            break
        policy = new
    rlhud.plot(values_at_start, label="v_π(start)", title="Policy iteration: value of start per iteration",
               xlabel="iteration")

    pi_star, V_star, iters = policy_iteration(env, gamma)
    rlhud.text(f"Policy iteration converged in {iters} iterations. v*(start) = {V_star[env.start]:.4f}",
               title="Done", tone="good")

## 04 · Value iteration

Policy iteration evaluates each policy *to convergence* before improving it. That seems
wasteful: after a few sweeps the greedy policy usually stops changing. Push the idea to the
extreme and **do just one sweep of evaluation, fused with improvement**:

```text
Vₖ₊₁(s) ← maxₐ Σₛ',ᵣ p(s', r | s, a) [ r + γ Vₖ(s') ]
```

That is the Bellman optimality equation turned into an update. It converges to v* from any
start (again a γ-contraction), and once V is close to v* you extract the policy with a single
one-step lookahead. The only difference from policy evaluation is **max instead of an average
over π**.

```python
for s in range(env.n_states):
    V[s] = max(sum(p * (r + (0 if done else gamma * V[s2])) for p, s2, r, done in env.P[s][a])
               for a in range(env.n_actions))
```

Step through value iteration below. Notice that the greedy arrows usually become optimal
long before the numbers stop changing. That is why stopping early (or using the policy
before V has fully converged) usually works well.

In [ ]:
#@title 🎮 **Interactive** · Dynamic programming sandbox
rlcolab.widget("gridworld", map=["....G", ".##.X", "....#", "S...."], gamma=0.95, step=-0.04, slip=0.2, mode="value-iteration")

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("04-dynamic-programming#2", {"q": "Value iteration on a world with γ = 0.9 has Δ = 1 after the first sweep. Roughly how many more sweeps until Δ < 10⁻⁶?", "options": ["About 6", "About 130", "About 10⁶", "It never gets below 10⁻⁶"], "answer": 1, "explain": "The error shrinks by at least a factor γ per sweep: 0.9ᵏ < 10⁻⁶ needs k > 6 / log₁₀(1/0.9) ≈ 131. Larger γ means slower convergence: γ = 0.99 needs about ten times as many sweeps."})

---
### 🧪 Exercise 3 · Value iteration on a slippery world · ⭐ **140 XP**

Turn the Bellman **optimality** equation into an update and solve a slippery maze.

`value_iteration(env, gamma, theta=1e-8, max_sweeps=10000)`:

1. Start from `V = 0`. In each sweep, for every state set `V[s] = max(q_from_v(env, V, s, gamma))` **in place**, tracking `delta`, the largest change.
2. Append each sweep's `delta` to a list; stop when `delta < theta` or after `max_sweeps` sweeps.
3. Extract the greedy policy as a one-hot `(n_states, n_actions)` array.

Return `(V, policy, deltas)`. `q_from_v` is provided.

**Run** solves a 10×5 maze where every move slips sideways 20% of the time, plots the convergence of Δ on a log scale, shows how good the greedy policy already is after only 3, 10 and 25 sweeps, and compares with the non-slippery version.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 04-dynamic-programming/03-value-iteration
import numpy as np
import rlhud
from rlenvs import GridWorld


def q_from_v(env, V, s, gamma):
    """One-step lookahead (from the MDP lesson): array of q(s, a) for every action."""
    q = np.zeros(env.n_actions)
    for a in range(env.n_actions):
        for prob, s2, r, done in env.P[s][a]:
            q[a] += prob * (r + (0.0 if done else gamma * V[s2]))
    return q


def value_iteration(env, gamma, theta=1e-8, max_sweeps=10_000):
    """Value iteration: sweep  V(s) ← max_a q(s, a)  over all states, in place.

    Track delta (largest change in the sweep) and stop when delta < theta, or after
    max_sweeps sweeps. Then extract the greedy policy as a one-hot array.

    Returns (V, policy, deltas): deltas is the list of per-sweep deltas
    (so len(deltas) is the number of sweeps).
    """
    V = np.zeros(env.n_states)
    deltas = []
    # TODO: in-place sweeps of V[s] = max_a q(s, a), recording each sweep's delta;
    #       stop when delta < theta (or after max_sweeps)
    raise NotImplementedError
    policy = np.zeros((env.n_states, env.n_actions))
    # TODO: one-hot greedy policy with respect to the final V
    return V, policy, deltas


if __name__ == "__main__":
    env = GridWorld(["......#..G",
                     ".####.#.#X",
                     ".#....#.#.",
                     ".#.##.....",
                     "S#....XX.."], slip=0.2, step_reward=-0.04)
    gamma = 0.99
    V, policy, deltas = value_iteration(env, gamma, theta=1e-6)
    rlhud.grid(env.to_grid(V), title=f"v* and π* · slip 0.2 · {len(deltas)} sweeps",
               policy=env.policy_grid(policy), walls=env.walls_mask(), labels=env.labels())
    rlhud.plot(np.log10(np.maximum(deltas, 1e-12)), label="log₁₀ Δ", title="Value iteration convergence",
               xlabel="sweep")

    # Truncated value iteration: how good is the greedy policy after only k sweeps?
    for k in [3, 10, 25]:
        Vk, pk, _ = value_iteration(env, gamma, max_sweeps=k)
        rlhud.grid(env.to_grid(Vk), title=f"After {k} sweeps", id=f"vi-{k}",
                   policy=env.policy_grid(pk), walls=env.walls_mask(), labels=env.labels())

    # Same world without slipping: every value is higher — uncertainty itself costs reward
    env0 = GridWorld(env.layout, slip=0.0, step_reward=-0.04)
    V0, p0, d0 = value_iteration(env0, gamma, theta=1e-6)
    rlhud.grid(env0.to_grid(V0), title="Deterministic version (slip 0)", policy=env0.policy_grid(p0),
               walls=env0.walls_mask(), labels=env0.labels())
    rlhud.stat("v*(start), slip 0.2", float(V[env.start]))
    rlhud.stat("v*(start), slip 0", float(V0[env0.start]))

In [ ]:
#@title ✅ **Check** · Value iteration on a slippery world (+140 XP) — tests the code you last ran above
rlcolab.check("04-dynamic-programming/03-value-iteration")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

Value iteration is policy evaluation with the average over π replaced by a max over actions. The loop skeleton is the same as in exercise 1.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

`for _ in range(max_sweeps): delta = 0.0; for s in range(env.n_states): v_new = q_from_v(env, V, s, gamma).max(); delta = max(delta, abs(v_new - V[s])); V[s] = v_new` then `deltas.append(delta)` and `if delta < theta: break`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Policy extraction after the loop: `for s in range(env.n_states): policy[s, np.argmax(q_from_v(env, V, s, gamma))] = 1.0`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("04-dynamic-programming/03-value-iteration")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 04-dynamic-programming/03-value-iteration
# Reference solution
import numpy as np
import rlhud
from rlenvs import GridWorld


def q_from_v(env, V, s, gamma):
    """One-step lookahead (from the MDP lesson): array of q(s, a) for every action."""
    q = np.zeros(env.n_actions)
    for a in range(env.n_actions):
        for prob, s2, r, done in env.P[s][a]:
            q[a] += prob * (r + (0.0 if done else gamma * V[s2]))
    return q


def value_iteration(env, gamma, theta=1e-8, max_sweeps=10_000):
    """Value iteration: sweep  V(s) ← max_a q(s, a)  over all states, in place.

    Track delta (largest change in the sweep) and stop when delta < theta, or after
    max_sweeps sweeps. Then extract the greedy policy as a one-hot array.

    Returns (V, policy, deltas): deltas is the list of per-sweep deltas
    (so len(deltas) is the number of sweeps).
    """
    V = np.zeros(env.n_states)
    deltas = []
    for _ in range(max_sweeps):
        delta = 0.0
        for s in range(env.n_states):
            v_new = q_from_v(env, V, s, gamma).max()
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        deltas.append(delta)
        if delta < theta:
            break
    policy = np.zeros((env.n_states, env.n_actions))
    for s in range(env.n_states):
        policy[s, np.argmax(q_from_v(env, V, s, gamma))] = 1.0
    return V, policy, deltas


if __name__ == "__main__":
    env = GridWorld(["......#..G",
                     ".####.#.#X",
                     ".#....#.#.",
                     ".#.##.....",
                     "S#....XX.."], slip=0.2, step_reward=-0.04)
    gamma = 0.99
    V, policy, deltas = value_iteration(env, gamma, theta=1e-6)
    rlhud.grid(env.to_grid(V), title=f"v* and π* · slip 0.2 · {len(deltas)} sweeps",
               policy=env.policy_grid(policy), walls=env.walls_mask(), labels=env.labels())
    rlhud.plot(np.log10(np.maximum(deltas, 1e-12)), label="log₁₀ Δ", title="Value iteration convergence",
               xlabel="sweep")

    # Truncated value iteration: how good is the greedy policy after only k sweeps?
    for k in [3, 10, 25]:
        Vk, pk, _ = value_iteration(env, gamma, max_sweeps=k)
        rlhud.grid(env.to_grid(Vk), title=f"After {k} sweeps", id=f"vi-{k}",
                   policy=env.policy_grid(pk), walls=env.walls_mask(), labels=env.labels())

    # Same world without slipping: every value is higher — uncertainty itself costs reward
    env0 = GridWorld(env.layout, slip=0.0, step_reward=-0.04)
    V0, p0, d0 = value_iteration(env0, gamma, theta=1e-6)
    rlhud.grid(env0.to_grid(V0), title="Deterministic version (slip 0)", policy=env0.policy_grid(p0),
               walls=env0.walls_mask(), labels=env0.labels())
    rlhud.stat("v*(start), slip 0.2", float(V[env.start]))
    rlhud.stat("v*(start), slip 0", float(V0[env0.start]))

## 05 · Generalized policy iteration

Policy iteration and value iteration are two points on one spectrum. Between them sits
*modified policy iteration*: evaluate with k sweeps, then improve. With k = 1 you get value
iteration; with k = ∞, policy iteration. You do not even need full sweeps: **asynchronous DP**
backs up states in any order, as many times as you like (focusing on states the agent actually
visits, say), and still converges as long as no state is ignored forever.

Sutton & Barto call the common pattern **generalized policy iteration (GPI)**: two processes
interact. Evaluation pulls V toward v_π for the current policy; improvement pulls π toward
greedy(V). Each process partially undoes the other (a new policy makes V stale; a new V makes
the policy non-greedy), yet together they converge on the only fixed point where both are
satisfied: v* and π*.

> **🔑 KEY IDEA** · **GPI is the template for all of RL**
>
> Almost every method in this course is GPI with cheaper, noisier ingredients. Monte Carlo and TD
> replace the model-based evaluation with estimates from sampled experience. SARSA and
> Q-learning interleave one-sample evaluation with ε-greedy improvement at every step.
> Actor-critic keeps an explicit critic (evaluation) and actor (improvement).

## 06 · The cost of needing a model

DP is efficient in a precise sense. One sweep costs O(|S|²·|A|) in the worst case (much less
when, as in grids, each action has only a few outcomes), and it finds π* in time polynomial in
|S| and |A|, even though there are |A|^|S| deterministic policies. That is exponentially better
than brute force. But it has two hard limits:

1. **It needs p(s', r | s, a).** For backgammon, a robot arm, or a game like Core Runner, nobody
   hands you the transition probabilities. You could estimate them, which is the idea behind
   model-based RL and Dyna-Q later, but usually you can only *sample* the environment by acting.
2. **It sweeps every state.** A 10 × 10 grid is 100 states; a 19 × 19 Go board has about 10¹⁷⁰.
   The number of states grows exponentially with the number of state variables, which Bellman
   called the **curse of dimensionality**.

The next lessons drop both requirements: Monte Carlo and TD methods learn from sampled
experience without any model, and only update the states the agent actually visits.

In [ ]:
#@title 🧠 **Knowledge check 4** · +25 XP — run, then pick an answer
rlcolab.quiz("04-dynamic-programming#3", {"q": "Which statement about DP is FALSE?", "options": ["Value iteration needs the full transition model", "Policy iteration converges in finitely many iterations on a finite MDP", "DP must evaluate all |A|^|S| policies to find the optimum", "In-place sweeps converge to the same values as two-array sweeps"], "answer": 2, "explain": "DP is polynomial in |S| and |A|. It never enumerates policies, which was the point of your brute-force exercise: to see what DP avoids."})

## 07 · Pitfalls

- **Bootstrapping through terminal transitions.** If `terminated` is True the target is r alone.
  Terminal states must stay at value 0.
- **Comparing floats for "policy stable".** Compare *actions* (`np.array_equal` on one-hot
  policies), never values, and keep old actions on ties.
- **Stopping on the wrong quantity.** Δ < θ says the values stopped changing, not that they
  are within θ of v*. With γ = 0.99 the true error can be about 100 times larger than Δ.
- **γ = 1 with a policy that never terminates.** Policy evaluation diverges (values go to −∞
  with step costs). Start policy iteration from a policy that reaches a terminal, such as the
  uniform random one, or use γ < 1.
- **Forgetting walls.** GridWorld numbers walls as states too. They are never reachable and
  their values stay 0. Use `env.walls_mask()` when plotting so they are not confused with real
  zeros.

## 08 · What carries forward

| DP idea | Model-free successor |
|---|---|
| Iterative policy evaluation | Monte Carlo prediction, TD(0) |
| Bellman optimality backup | Q-learning, DQN target r + γ maxₐ' Q(s', a') |
| Greedy policy improvement | ε-greedy control, policy-gradient updates |
| Generalized policy iteration | SARSA, Q-learning, actor-critic |
| Exact v* on small worlds | Ground truth for testing every learner you build |

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("04-dynamic-programming")